In [2]:
import nltk
from nltk.corpus import treebank
from nltk.tag import hmm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction import DictVectorizer
import random

# Download necessary NLTK data
# try:
#     nltk.data.find('corpora/treebank')
#     nltk.data.find('corpora/universal_tagset')
# except nltk.downloader.DownloadError:
nltk.download('treebank')
nltk.download('universal_tagset')

# --- Step 1: Load and Split the data ---
sentences = list(treebank.tagged_sents(tagset='universal'))
random.shuffle(sentences)

train_data = sentences[:3000]
test_data = sentences[3000:]

# --- Step 2: Implement HMM with Viterbi Decoding ---
print("--- HMM Model ---")
trainer = hmm.HiddenMarkovModelTrainer()
hmm_tagger = trainer.train(train_data)
hmm_accuracy = hmm_tagger.evaluate(test_data)
print(f"HMM (Viterbi) Accuracy: {hmm_accuracy:.4f}\n")

# --- Step 3 & 4: Extract Features and Train Log-linear Model ---
print("--- Log-linear Model ---")

# Feature extraction function as defined in the notes
def extract_features(sentence, i):
    word = sentence[i][0]
    features = {
        'word': word,
        'is_capitalized': word[0].isupper(),
        'is_digit': word.isdigit(),
        'prefix-1': word[0],
        'suffix-1': word[-1],
        'suffix-2': word[-2:],
    }
    if i > 0:
        features['prev_word'] = sentence[i-1][0]
    else:
        features['prev_word'] = '<START>'
    return features

# Function to prepare dataset for sklearn
def prepare_dataset(tagged_sents):
    X, y = [], []
    for sent in tagged_sents:
        words, tags = zip(*sent)
        for i in range(len(words)):
            feats = extract_features(sent, i)
            X.append(feats)
            y.append(tags[i])
    return X, y

# Prepare the data
X_train_dict, y_train = prepare_dataset(train_data)
X_test_dict, y_test = prepare_dataset(test_data)

# Vectorize the dictionary features
vectorizer = DictVectorizer()
X_train_vec = vectorizer.fit_transform(X_train_dict)
X_test_vec = vectorizer.transform(X_test_dict)

# Train the Logistic Regression model
clf = LogisticRegression(max_iter=200, solver='lbfgs') # Increased max_iter for convergence
clf.fit(X_train_vec, y_train)

# Predict and evaluate
y_pred = clf.predict(X_test_vec)
log_linear_accuracy = accuracy_score(y_test, y_pred)
print(f"Log-linear Model Accuracy: {log_linear_accuracy:.4f}\n")

# --- Step 5: Performance Comparison ---
print("--- Performance Comparison ---")
print(f"HMM (Viterbi): {hmm_accuracy:.4f}")
print(f"Log-linear Model: {log_linear_accuracy:.4f}")

[nltk_data] Downloading package treebank to /root/nltk_data...
[nltk_data]   Unzipping corpora/treebank.zip.
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.


--- HMM Model ---


/tmp/ipython-input-1486884059.py:28: DeprecationWarning: 
  Function evaluate() has been deprecated.  Use accuracy(gold)
  instead.
  hmm_accuracy = hmm_tagger.evaluate(test_data)
/usr/local/lib/python3.12/dist-packages/nltk/tag/hmm.py:335: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])
/usr/local/lib/python3.12/dist-packages/nltk/tag/hmm.py:363: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])


HMM (Viterbi) Accuracy: 0.5775

--- Log-linear Model ---
Log-linear Model Accuracy: 0.9605

--- Performance Comparison ---
HMM (Viterbi): 0.5775
Log-linear Model: 0.9605
